# Employee Attrition Prediction - Complete Analysis

**Analyst:** Balla Prem Kumar | **Date:** June 26, 2026

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScalerfrom sklearn.linear_model import LogisticRegressionfrom sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifierfrom sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve, aucimport warningswarnings.filterwarnings('ignore')print('All libraries imported successfully!')

# TASK 1: Data Loading & Exploration

In [ ]:
# GitHub raw URL - CSV already in repocsv_url = 'https://raw.githubusercontent.com/Prem-0007/EmployeeAttrition_BALLA_PREM_KUMAR/main/WA_Fn-UseC_-HR-Employee-Attrition.csv'# Load datasetdf = pd.read_csv(csv_url)print(f'Dataset Shape: {df.shape}')print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')print(df.head(10))

In [ ]:
# Analyze attrition targetprint('ATTRITION ANALYSIS:')print(df['Attrition'].value_counts())# Calculate ratestotal = len(df)left = (df['Attrition'] == 'Yes').sum()stayed = (df['Attrition'] == 'No').sum()rate = (left/total)*100print(f'Total: {total}, Left: {left}, Stayed: {stayed}')print(f'Attrition Rate: {rate:.2f}%')print(f'Status: CLASS IMBALANCE - Only {rate:.1f}% left (minority class)')

# TASK 2: Data Cleaning & Preprocessing

In [ ]:
# Drop irrelevant columnsdf_clean = df.copy()df_clean = df_clean.drop(columns=['EmployeeNumber', 'Over18', 'StandardHours'], errors='ignore')# Convert target to binarydf_clean['Attrition'] = (df_clean['Attrition'] == 'Yes').astype(int)# Get categorical columns AFTER droppingcategorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()# One-hot encodedf_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)# Scale numeric featuresnumeric_features = df_encoded.select_dtypes(include=['int64', 'float64']).columns.tolist()numeric_features.remove('Attrition')scaler = StandardScaler()df_encoded[numeric_features] = scaler.fit_transform(df_encoded[numeric_features])print(f'Data prepared! Shape: {df_encoded.shape}')

# TASK 3: Exploratory Data Analysis

In [ ]:
# Reload original for EDAdf_eda = pd.read_csv(csv_url)df_eda = df_eda.drop(columns=['EmployeeNumber', 'Over18', 'StandardHours'], errors='ignore')df_eda['Attrition_Binary'] = (df_eda['Attrition'] == 'Yes').astype(int)# Department analysisdept = df_eda.groupby('Department')['Attrition_Binary'].agg(['sum', 'count'])dept['Rate'] = (dept['sum']/dept['count']*100).round(2)print('ATTRITION BY DEPARTMENT:')print(dept.sort_values('Rate', ascending=False))# Income analysisleft_income = df_eda[df_eda['Attrition']=='Yes']['MonthlyIncome'].mean()stayed_income = df_eda[df_eda['Attrition']=='No']['MonthlyIncome'].mean()diff_pct = ((stayed_income - left_income) / left_income) * 100print(f'\nINCOME: Left earn Rs{left_income:,.0f}, Stayed earn Rs{stayed_income:,.0f}')print(f'Difference: {diff_pct:.1f}% lower for those who left')# Work-life balancewlb = df_eda.groupby('WorkLifeBalance')['Attrition_Binary'].agg(['sum', 'count'])wlb['Rate'] = (wlb['sum']/wlb['count']*100).round(2)print(f'\nWORK-LIFE BALANCE (1=Bad, 4=Best):')print(wlb[['Rate']])print(f'GAP: {wlb.loc[1, "Rate"] - wlb.loc[4, "Rate"]:.1f} percentage points!')

# TASK 4-5: Model Building & Evaluation

In [ ]:
# Prepare dataX = df_encoded.drop('Attrition', axis=1)y = df_encoded['Attrition']# Train-test splitX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')# Train 3 modelsprint('\nTraining models...')lr = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)lr.fit(X_train, y_train)print('✓ Logistic Regression')rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)rf.fit(X_train, y_train)print('✓ Random Forest')gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)gb.fit(X_train, y_train)print('✓ Gradient Boosting')

In [ ]:
# Evaluatepredictions = {    'Logistic Regression': (lr.predict(X_test), lr.predict_proba(X_test)[:, 1]),    'Random Forest': (rf.predict(X_test), rf.predict_proba(X_test)[:, 1]),    'Gradient Boosting': (gb.predict(X_test), gb.predict_proba(X_test)[:, 1])}results = []for name, (pred, proba) in predictions.items():    results.append({        'Model': name,        'Precision': precision_score(y_test, pred),        'Recall': recall_score(y_test, pred),        'F1-Score': f1_score(y_test, pred),        'ROC-AUC': roc_auc_score(y_test, proba)    })results_df = pd.DataFrame(results).round(4)print('\nMODEL COMPARISON:')print(results_df.to_string(index=False))best_idx = results_df['F1-Score'].idxmax()best_model = results_df.loc[best_idx, 'Model']print(f'\n✓ BEST MODEL: {best_model}')

In [ ]:
# Feature importanceif best_model == 'Random Forest':    importance = rf.feature_importances_elif best_model == 'Gradient Boosting':    importance = gb.feature_importances_else:    importance = np.abs(lr.coef_[0])feat_imp = pd.DataFrame({'Feature': X.columns, 'Importance': importance})feat_imp = feat_imp.sort_values('Importance', ascending=False)print('TOP 10 FEATURES:')print(feat_imp.head(10).to_string(index=False))

# TASK 6: Visualizations

In [ ]:
# Chart 1fig, ax = plt.subplots(figsize=(10, 5))dept_rates = dept.sort_values('Rate')ax.barh(dept_rates.index, dept_rates['Rate'], color=['#2ca02c', '#ff7f0e', '#d62728'])ax.set_xlabel('Attrition Rate (%)')ax.set_title('Attrition by Department')ax.grid(axis='x', alpha=0.3)for i, v in enumerate(dept_rates['Rate']):    ax.text(v+0.5, i, f'{v:.1f}%', va='center', fontweight='bold')plt.tight_layout()plt.savefig('chart_1_attrition_by_dept_role.png', dpi=150, bbox_inches='tight')plt.show()print('✓ Chart 1 saved')

In [ ]:
# Chart 2fig, ax = plt.subplots(figsize=(10, 5))left_data = df_eda[df_eda['Attrition']=='Yes']['MonthlyIncome']stayed_data = df_eda[df_eda['Attrition']=='No']['MonthlyIncome']bp = ax.boxplot([left_data, stayed_data], tick_labels=['Left', 'Stayed'], patch_artist=True)for patch, color in zip(bp['boxes'], ['#d62728', '#2ca02c']):    patch.set_facecolor(color)ax.set_ylabel('Monthly Income (Rs)')ax.set_title('Income: Left vs Stayed')plt.tight_layout()plt.savefig('chart_2_income_comparison.png', dpi=150, bbox_inches='tight')plt.show()print('✓ Chart 2 saved')

In [ ]:
# Chart 3fig, ax = plt.subplots(figsize=(8, 6))cm = confusion_matrix(y_test, predictions[best_model][0])sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True, ax=ax)ax.set_title(f'Confusion Matrix: {best_model}')plt.tight_layout()plt.savefig('chart_3_confusion_matrix.png', dpi=150, bbox_inches='tight')plt.show()print('✓ Chart 3 saved')

In [ ]:
# Chart 4fig, ax = plt.subplots(figsize=(10, 6))top_10 = feat_imp.head(10).sort_values('Importance')ax.barh(top_10['Feature'], top_10['Importance'], color='#2ca02c')ax.set_xlabel('Importance')ax.set_title('Top 10 Features')plt.tight_layout()plt.savefig('chart_4_top_10_features.png', dpi=150, bbox_inches='tight')plt.show()print('✓ Chart 4 saved')

In [ ]:
# Chart 5 - ROC curvesfig, ax = plt.subplots(figsize=(10, 8))for name, (pred, proba) in predictions.items():    fpr, tpr, _ = roc_curve(y_test, proba)    auc_score = auc(fpr, tpr)    ax.plot(fpr, tpr, label=f'{name} (AUC={auc_score:.4f})', linewidth=2)ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')ax.set_xlabel('False Positive Rate')ax.set_ylabel('True Positive Rate')ax.set_title('ROC Curves: Model Comparison')ax.legend(loc='lower right')ax.grid(alpha=0.3)plt.tight_layout()plt.savefig('chart_5_roc_curves.png', dpi=150, bbox_inches='tight')plt.show()print('✓ Chart 5 saved - ALL VISUALIZATIONS COMPLETE!')

# TASK 7: HR Insights & Recommendations

In [ ]:
print('='*80)print('KEY FINDINGS & RECOMMENDATIONS')print('='*80)print(f'\nTop Risk Factors (from ML analysis):')for idx, (i, row) in enumerate(feat_imp.head(3).iterrows(), 1):    print(f'{idx}. {row["Feature"]}')print(f'\nHighest Risk: {dept.idxmax()['Rate']} Department ({dept['Rate'].max():.1f}% attrition)')print(f'\nFirst-year attrition: {df_eda[df_eda["YearsAtCompany"]==1]["Attrition_Binary"].mean()*100:.1f}%')print(f'\nRECOMMENDATIONS:')print('1. First-Year Engagement Program (33% attrition risk)')print('   - Mentor assignment for 6 months')print('   - Monthly check-ins with managers')print('   - 90-day role clarity conversation')print(f'\n2. {dept.idxmax()['Rate']} Department Retention')print('   - Review compensation structure')print('   - Create clear career paths')print('   - Improve work-life balance')print('\n3. MODEL LIMITATIONS:')print('   - Based on historical data')   - Cannot predict sudden unexpected departures')   - Correlation does not equal causation')   - Use for informed decisions, not automated actions')print('='*80)